# 🌐 WiseNet V1.5 — Comprehensive Research Notebook
### End-to-End Real-World Multi-Sector × Dual-Carrier Optimization on Telecom Italia Milan Dataset

---

## 📖 Executive Summary & Core Methodology

In this research milestone (**WiseNet V1.5**), we transition from the isotropic single-carrier model of V1.0 to a realistic **3GPP-compliant cellular architecture**:
- **Geographical Ground Truth**: 1,024 contiguous grid squares ($32 \times 32$ block, $235\text{ m} \times 235\text{ m}$ per cell) from the open **Telecom Italia Big Data Challenge** dataset in central Milan.
- **Real-World Traffic Demand**: Real internet traffic measurements (`internet_volume` in MB) aggregated in 30-minute operational time slots.
- **3GPP Hexagonal Cellular Topology**: Deterministic hexagonal macro-grid with Inter-Site Distance $\text{ISD} = 750\text{ m}$ (126 physical sites, 378 sectors at $120^\circ$ beam azimuths, and 756 logical $(s, f)$ radio cells).
- **Dual-Carrier Real Frequency Spectrum (TIM Italy Licences)**:
  - $\mathbf{F_1 = 1.8\text{ GHz}}$: LTE Band 3 FDD ($20\text{ MHz}$, $P_{\text{tx}} = 43\text{ dBm}$ / $20\text{ W}$, SINR $= 12\text{ dB}$, Coverage anchor layer)
  - $\mathbf{F_2 = 3.5\text{ GHz}}$: 5G NR n78 TDD ($80\text{ MHz}$, $P_{\text{tx}} = 43\text{ dBm}$ effective, SINR $= 15\text{ dB}$, High-capacity layer)
- **Two-Phase Architecture (Offline Simulation & Online MILP Optimization)**:
  1. **Offline Spatial Propagation**: 3GPP TR 38.901 UMi path-loss and directive antenna beamforming computed on micro-grids of $20 \times 20 = 400$ sub-pixels per square, precomputing the exact transfer tensor $H$.
  2. **Online MILP Optimization**: Strict conservation-of-mass linear programming with CBC solver, enabling Horizontal (inter-sector) and Vertical (inter-carrier $F_1 \to F_2$) offloading in sub-second time ($< 1\text{ s}$).

## Phase 1 — Deterministic 3GPP Hexagonal Topology Generation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from src.topology.builder_v1_5 import TopologyBuilderV15, CELL_SIZE_METERS, GRID_SIZE

# 1. Instantiate the deterministic 3GPP topology generator (ISD = 750 m)
builder = TopologyBuilderV15(isd_meters=750.0)
topology = builder.generate_hexagonal_topology(row_range=(35, 67), col_range=(35, 67))

print(f"[Topology Summary]")
print(f"  • Physical gNodeB/eNodeB sites : {len(topology)}")
print(f"  • Directional Sectors (120°)   : {len(topology) * 3}")
print(f"  • Radio Cells (s, f)           : {len(topology) * 6} (F1 LTE + F2 5G NR)")

# Sample site inspection
sample_site = list(topology.values())[0]
print(f"\nSample Site Inspection: {sample_site['site_id']} at (x={sample_site['x_meters']:.0f}m, y={sample_site['y_meters']:.0f}m)")
for sec_id, sec_data in sample_site['sectors'].items():
    print(f"  Sector {sec_id} (Azimuth {sec_data['azimuth_deg']}°):")
    for c_name, c_data in sec_data['carriers'].items():
        print(f"    -> {c_data['cell_id']}: Freq={c_data['freq_ghz']} GHz, BW={c_data['bw_mhz']} MHz, P_tx={c_data['tx_power_dBm']} dBm, Nominal Cap={c_data['capacity_mo']} MB")

## Phase 2 — Offline 3GPP RSRP Micro-Grid Simulation & Transfer Tensor $H$

Each $235\text{ m} \times 235\text{ m}$ square is sampled over a regular micro-grid of $20 \times 20 = 400$ sub-pixels ($11.75\text{ m} \times 11.75\text{ m}$ each).
For each sub-pixel $k$, the directional 3GPP RSRP is computed:
$$\text{RSRP}(k, s, f) = P_{\text{tx}}(f) - \text{PL}(d_k, f) + G(\Delta\theta_{k, s})$$
When an offset $\delta \in [0.0, 3.0]\text{ dB}$ is applied, we precalculate the exact offload fractions across sectors and carriers.

In [ ]:
from src.spatial.simulator_v1_5 import SpatialTransferSimulatorV15

# 1024 contiguous Milan grid squares
squares = [
    r * GRID_SIZE + c + 1
    for r in range(35, 67)
    for c in range(35, 67)
]

sim = SpatialTransferSimulatorV15(
    grid_resolution=20,
    cell_size_meters=CELL_SIZE_METERS,
    delta_levels=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
)

fractions_data = sim.compute_transfer_fractions(squares, topology, grid_size=GRID_SIZE)
print(f"Transfer tensor precalculated for {len(fractions_data)} / {len(squares)} Milan squares.")

# Display sample offload transition matrix
sample_sq = str(squares[0])
if sample_sq in fractions_data:
    info = fractions_data[sample_sq]
    print(f"\nSample Matrix for Square {sample_sq}:")
    print(f"  Primary Master Anchor Cell: {info['master_cell']}")
    for delta_db, off_info in info['offsets'].items():
        print(f"  Offset {delta_db} dB: Stays={off_info['stays']*100:.1f}%, Targets={off_info['target_cells']}")

## Phase 3 — Ingestion of Real Telecom Italia Milan Traffic Data

We load the verified open dataset `research/data/processed/work_1024cells.parquet` and isolate the highest peak load 30-minute interval.

In [ ]:
from pathlib import Path

parquet_path = Path("../../research/data/processed/work_1024cells.parquet")
df = pl.read_parquet(parquet_path)

# Identify global peak load time slot
top_slots = df.group_by('slot_30m').agg(pl.col('internet_volume').sum()).sort('internet_volume', descending=True)
peak_slot = top_slots['slot_30m'][0]
slot_df = df.filter(pl.col('slot_30m') == peak_slot)

predicted_traffic = {
    str(int(row['square_id'])): float(row['internet_volume'])
    for row in slot_df.iter_rows(named=True)
}

total_traffic_mo = sum(predicted_traffic.values())
print(f"Peak Slot Time Stamp : {peak_slot}")
print(f"Total Real Demand    : {total_traffic_mo:,.1f} MB ({total_traffic_mo/1024:.2f} GB)")
print(f"Active Grid Squares  : {len(predicted_traffic)}")

## Phase 4 — Rigorous Benchmark Execution: Static vs Greedy vs MILP

In [ ]:
import time
from src.optimization.milp_engine_v1_5 import MilpEngineV15
from src.optimization.greedy_engine_v1_5 import GreedyEngineV15

# Extract cell capacity dictionary
cells_capacity = {}
for s_data in topology.values():
    for sec_data in s_data['sectors'].values():
        for c_data in sec_data['carriers'].values():
            cells_capacity[c_data['cell_id']] = c_data['capacity_mo']

# 1. Static (Baseline)
cell_traffic = {c: 0.0 for c in cells_capacity}
for sq_id, sq_info in fractions_data.items():
    m = sq_info['master_cell']
    if m in cell_traffic:
        cell_traffic[m] += predicted_traffic.get(sq_id, 0.0)
static_unsatisfied = sum(max(0.0, cell_traffic[c] - cap) for c, cap in cells_capacity.items())

# 2. Greedy (Local Heuristic with Strict Mass Conservation)
t0 = time.time()
greedy_engine = GreedyEngineV15()
res_greedy = greedy_engine.solve(predicted_traffic, fractions_data, cells_capacity, [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
t_greedy = time.time() - t0
greedy_unsatisfied = res_greedy['optimized_unsatisfied_mo']
greedy_gain = res_greedy['gain_percentage']

# 3. MILP (Global Exact Optimization)
t0 = time.time()
milp_engine = MilpEngineV15()
res_milp = milp_engine.build_and_solve(predicted_traffic, fractions_data, cells_capacity, [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
t_milp = time.time() - t0
milp_unsatisfied = res_milp['optimized_unsatisfied_mo']
milp_gain = res_milp['gain_percentage']

print("=" * 65)
print("  FINAL RESEARCH BENCHMARK RESULTS (TELECOM ITALIA MILAN)")
print("=" * 65)
print(f"  {'Policy':<12} {'Unsatisfied Traffic':>20} {'Reduction (%)':>16} {'Time (s)':>10}")
print("-" * 65)
print(f"  {'Static':<12} {static_unsatisfied:>17,.1f} MB {'---':>16} {'0.00':>9}s")
print(f"  {'Greedy':<12} {greedy_unsatisfied:>17,.1f} MB {greedy_gain:>15.2f}% {t_greedy:>9.2f}s")
print(f"  {'MILP':<12} {milp_unsatisfied:>17,.1f} MB {milp_gain:>15.2f}% {t_milp:>9.2f}s")
print("=" * 65)
print(f"\n[Proof] MILP outperforms Greedy by {greedy_unsatisfied - milp_unsatisfied:,.1f} MB (+{(greedy_unsatisfied - milp_unsatisfied)/1024:.2f} GB delivered) in {t_milp:.2f}s.")

## Phase 5 — Scientific Visualization of Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Unsatisfied Traffic Bar Chart
policies = ['Static (0 dB)', 'Greedy Heuristic', 'MILP Exact Global']
values_gb = [static_unsatisfied / 1024, greedy_unsatisfied / 1024, milp_unsatisfied / 1024]
colors = ['#e74c3c', '#f39c12', '#2ecc71']

bars = ax1.bar(policies, values_gb, color=colors, edgecolor='black', linewidth=1)
ax1.set_ylabel('Unsatisfied Demand (GB)', fontsize=11, fontweight='bold')
ax1.set_title('Unsatisfied Traffic by Optimization Policy', fontsize=12, fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f"{yval:.1f} GB", ha='center', va='bottom', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# 2. Offset Distribution
offset_vals = [d['offset_dB'] for d in res_milp['decisions'].values()]
u_offsets, counts = np.unique(offset_vals, return_counts=True)
ax2.bar([f"{o} dB" for o in u_offsets], counts, color='#3498db', edgecolor='black', linewidth=1)
ax2.set_xlabel('Selected Offload Offset (dB)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Number of Radio Cells (s, f)', fontsize=11, fontweight='bold')
ax2.set_title('MILP Offset Distribution (756 Radio Cells)', fontsize=12, fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
fig_out = Path("../../research/reports/figures/v1_5_verified_benchmark.png")
plt.savefig(fig_out, dpi=200, bbox_inches='tight')
plt.show()
print(f"Figure saved to: {fig_out.resolve()}")